In [9]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [10]:
class orderprocessingState(TypedDict):
  customer:str
  amount:int
  payment_done:bool
  payment_status:str
  order_type:str
  result:str


In [11]:
def check_payment(state: orderprocessingState):
    if state["payment_done"] == True:
        state["payment_status"] = "Paid"
    else:
        state["payment_status"] = "Pending"

    return state


def check_order(state: orderprocessingState):
    if state["amount"] >= 2000:
        state["order_type"] = "Premium"
    else:
        state["order_type"] = "Basic"

    return state


def router(state: orderprocessingState):
    if state["payment_status"] == "Pending":
        return "payment_pending"

    elif state["order_type"] == "Premium":
        return "premium_order"

    else:
        return "regular_order"


def payment_pending(state: orderprocessingState):
    state["result"] = "Order cannot be processed because payment is pending"

    return state


def premium_order(state: orderprocessingState):
    state["result"] = "Premium order processed successfully."

    return state


def regular_order(state: orderprocessingState):
    state["result"] = "Regular order processed successfully."

    return state


graph = StateGraph(orderprocessingState)

In [12]:
graph.add_node("check_payment", check_payment)
graph.add_node("check_order", check_order)
graph.add_node("payment_pending", payment_pending)
graph.add_node("premium_order", premium_order)
graph.add_node("regular_order", regular_order)


In [13]:
graph.add_edge(START, "check_payment")

graph.add_edge("check_payment", "check_order")


# 11. Add conditional routing
graph.add_conditional_edges(
    "check_order",
    router,
    {
        "payment_pending": "payment_pending",
        "premium_order": "premium_order",
        "regular_order": "regular_order"
    }
)
graph.add_edge("payment_pending", END)
graph.add_edge("premium_order", END)
graph.add_edge("regular_order", END)
app = graph.compile()

In [14]:
input_data = {
    "customer": "Anushka",
    "amount": 2500,
    "payment_done": True,
    "payment_status": "",
    "order_type": "",
    "result": ""
}

In [15]:
print(input_data.keys())

dict_keys(['customer', 'amount', 'payment_done', 'payment_status', 'order_type', 'result'])


In [16]:
result = app.invoke(input_data)

print(result)
print(result["result"])

{'customer': 'Anushka', 'amount': 2500, 'payment_done': True, 'payment_status': 'Paid', 'order_type': 'Premium', 'result': 'Premium order processed successfully.'}
Premium order processed successfully.
